In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Load dataset
df = pd.read_csv('car_data.csv')

# Inspect first few rows
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())

   Unnamed: 0       car_name    brand     model  vehicle_age  km_driven  \
0           0    Maruti Alto   Maruti      Alto            9     120000   
1           1  Hyundai Grand  Hyundai     Grand            5      20000   
2           2    Hyundai i20  Hyundai       i20           11      60000   
3           3    Maruti Alto   Maruti      Alto            9      37000   
4           4  Ford Ecosport     Ford  Ecosport            6      30000   

  seller_type fuel_type transmission_type  mileage  engine  max_power  seats  \
0  Individual    Petrol            Manual    19.70     796      46.30      5   
1  Individual    Petrol            Manual    18.90    1197      82.00      5   
2  Individual    Petrol            Manual    17.00    1197      80.00      5   
3  Individual    Petrol            Manual    20.92     998      67.10      5   
4      Dealer    Diesel            Manual    22.77    1498      98.59      5   

   selling_price  
0         120000  
1         550000  
2         2

In [3]:
# Cell 2: Preprocessing & Clean Feature Setup

# 1. Inspect actual column names to handle case sensitivity
print("Columns in dataset:", df.columns.tolist())

# Clean up column names (strip whitespace and convert to lowercase)
df.columns = df.columns.str.strip().str.lower()

# 2. Derive or assign Vehicle_Age
if 'year' in df.columns:
    df['vehicle_age'] = 2026 - df['year']
elif 'vehicle_age' in df.columns:
    df['vehicle_age'] = df['vehicle_age']

# 3. Drop unneeded index and non-numeric textual columns (car_name, brand, model)
cols_to_drop = [col for col in ['unnamed: 0', 'car_name', 'brand', 'model', 'year'] if col in df.columns]
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# 4. One-Hot Encoding for categorical features
df = pd.get_dummies(df, drop_first=True)

print("\nProcessed Features Available:")
print(df.columns.tolist())

Columns in dataset: ['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'seller_type', 'fuel_type', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price']

Processed Features Available:
['vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power', 'seats', 'selling_price', 'seller_type_Individual', 'seller_type_Trustmark Dealer', 'fuel_type_Diesel', 'fuel_type_Electric', 'fuel_type_LPG', 'fuel_type_Petrol', 'transmission_type_Manual']


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

# Separate features (X) and target variable (y)
target_col = 'selling_price' if 'selling_price' in df.columns else 'Selling_Price'

X = df.drop(columns=[target_col])
y = df[target_col]

# 80-20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Save feature names for Streamlit deployment
joblib.dump(X.columns.tolist(), 'model_features.pkl')

# Train and compare 3 models
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    results.append({
        "Model": name, 
        "R2 Score": round(r2, 4), 
        "MAE": round(mae, 4), 
        "RMSE": round(rmse, 4)
    })

results_df = pd.DataFrame(results)
print(results_df)

               Model  R2 Score          MAE         RMSE
0  Linear Regression    0.6651  278281.6622  502087.8408
1      Decision Tree    0.8522  133527.2381  333547.8284
2      Random Forest    0.9255  105036.4097  236763.6289


In [5]:
# Train best model (Random Forest) and serialize
best_model = RandomForestRegressor(n_estimators=100, random_state=42)
best_model.fit(X_train, y_train)

joblib.dump(best_model, 'car_price_model.pkl')
print("Saved car_price_model.pkl and model_features.pkl successfully!")

Saved car_price_model.pkl and model_features.pkl successfully!
